# 均线策略回测系统

## 功能说明

本 Notebook 实现了一个完整的双均线策略回测系统，包含：

1. **数据加载**：加载5只不同行业的复权股价数据
2. **技术指标计算**：MA/EMA、ATR
3. **交易信号生成**：双均线策略（金叉/死叉）+ 趋势过滤器 + ATR过滤器
4. **回测执行**：模拟交易，计算净值曲线
5. **量化指标计算**：收益率、夏普比率、最大回撤、胜率等
6. **可视化**：K线图、均线、交易信号、净值曲线
7. **BUY-HOLD 对比**：与买入持有策略比较

---


In [ ]:
# Cell 1: 环境配置与依赖检查
import sys
import subprocess

# 检查并安装依赖
def install_package(package):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', package])

required_packages = ['pandas', 'numpy', 'plotly', 'matplotlib']

for package in required_packages:
    try:
        __import__(package)
        print(f'{package} 已安装')
    except ImportError:
        print(f'安装 {package}...')
        install_package(package)

print('\n依赖检查完成！')

In [ ]:
# Cell 2: 导入库与模块
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import matplotlib.pyplot as plt
from datetime import datetime
import warnings
import os
warnings.filterwarnings('ignore')

# 导入策略模块
sys.path.append('.')
from strategy import (
    generate_signals,
    run_backtest,
    run_buy_hold_backtest,
    calculate_metrics,
    format_metrics_report
)

print('库导入成功！')
print('可用模块：')
print('  - generate_signals(): 生成交易信号')
print('  - run_backtest(): 执行回测')
print('  - calculate_metrics(): 计算量化指标')

In [ ]:
# Cell 3: 策略参数配置（用户可自定义）

STRATEGY_PARAMS = {
    'short_window': 5,           # 短均线周期
    'long_window': 15,           # 长均线周期
    'ma_type': 'MA',             # 'MA' 或 'EMA'
    'trend_filter': True,        # 是否启用趋势过滤器
    'trend_window': 120,        # 趋势过滤器周期
    'atr_filter': True,          # 是否启用ATR过滤器
    'atr_window': 14,           # ATR计算周期
    'atr_percentile': 20,       # ATR历史百分位阈值（P20）
    'atr_lookback': 100,       # ATR历史lookback天数
    'initial_capital': 100000,   # 初始资金（元）
    'commission': 0.001,        # 手续费率（0.1% = 0.001）
    'slippage': 0.001,         # 滑点（0.1% = 0.001）
    'position_sizing': 'full',   # 仓位管理方式：'full' | 'fixed_shares' | 'fixed_ratio'
    'fixed_shares': 100,       # 固定数量
    'fixed_ratio': 0.2,         # 固定比例
    'start_date': '2024-07-03',
    'end_date': '2026-07-03',
}

print('策略参数配置完成！')
print('=' * 60)
for key, value in STRATEGY_PARAMS.items():
    print(f'{key}: {value}')
print('=' * 60)

In [ ]:
# Cell 4: 数据加载函数

def load_stock_data(ts_code, start_date=None, end_date=None):
    """
    加载股票数据
    
    参数：
        ts_code: 股票代码
        start_date: 起始日期（可选）
        end_date: 结束日期（可选）
    
    返回：
        处理后的DataFrame
    """
    # 根据ts_code找到对应的文件名
    file_map = {
        '300750.SZ': '宁德时代_300750_daily_adjusted.csv',
        '601318.SH': '中国平安_601318_daily_adjusted.csv',
        '600519.SH': '贵州茅台_600519_daily_adjusted.csv',
        '601857.SH': '中国石油_601857_daily_adjusted.csv',
        '002594.SZ': '比亚迪_002594_daily_adjusted.csv'
    }
    
    filename = file_map.get(ts_code)
    if not filename:
        raise ValueError(f'未找到股票代码 {ts_code} 对应的文件')
    
    filepath = os.path.join('data/adjusted', filename)
    
    if not os.path.exists(filepath):
        raise FileNotFoundError(f'文件不存在: {filepath}')
    
    # 加载数据
    df = pd.read_csv(filepath)
    
    # 数据预处理
    df['trade_date'] = pd.to_datetime(df['trade_date'])
    df = df.sort_values('trade_date').reset_index(drop=True)
    
    # 筛选日期范围
    if start_date:
        start_date = pd.to_datetime(start_date)
        df = df[df['trade_date'] >= start_date].copy()
    if end_date:
        end_date = pd.to_datetime(end_date)
        df = df[df['trade_date'] <= end_date].copy()
    
    # 重置索引
    df = df.reset_index(drop=True)
    
    print(f'加载数据: {filename}')
    print(f'  日期范围: {df["trade_date"].min().date()} 至 {df["trade_date"].max().date()}')
    print(f'  数据条数: {len(df)}')
    
    return df

print('数据加载函数定义完成！')
print('使用方法：df = load_stock_data("600519.SH")')

In [ ]:
# Cell 5: 加载示例数据（贵州茅台）

# 选择一只股票进行演示
STOCK_CODE = '600519.SH'
STOCK_NAME = '贵州茅台'
STOCK_INDUSTRY = '消费'

# 加载数据
df = load_stock_data(
    STOCK_CODE,
    start_date=STRATEGY_PARAMS['start_date'],
    end_date=STRATEGY_PARAMS['end_date']
)

# 查看数据前5行
print('\n数据预览：')
display(df.head())

In [ ]:
# Cell 6: 计算技术指标并生成交易信号

print('计算技术指标和生成信号...')
df = generate_signals(df, STRATEGY_PARAMS)

# 统计信号数量
num_buy = (df['signal'] == 1).sum()
num_sell = (df['signal'] == -1).sum()
print(f'生成买入信号: {num_buy} 个')
print(f'生成卖出信号: {num_sell} 个')

# 查看计算结果前10行
print('\n计算结果预览（前10行）：')
display_cols = ['trade_date', 'close', 'short_ma', 'long_ma', 'atr', 'atr_percentile', 'signal']
display(df[display_cols].head(10))

In [ ]:
# Cell 7: 执行回测

print('执行策略回测...')
backtest_result = run_backtest(df, STRATEGY_PARAMS)
print(f'策略最终净值: {backtest_result["final_value"]:.2f} 元')
print(f'策略交易次数: {len(backtest_result["trades"])}')

print('\n执行BUY-HOLD回测...')
buy_hold_result = run_buy_hold_backtest(df, STRATEGY_PARAMS)
print(f'BUY-HOLD最终净值: {buy_hold_result["final_value"]:.2f} 元')

# 查看交易记录
if len(backtest_result['trades']) > 0:
    print('\n交易记录（前10条）：')
    display(backtest_result['trades'].head(10))

In [ ]:
# Cell 8: 计算量化指标

print('计算量化指标...')
metrics = calculate_metrics(backtest_result, buy_hold_result, df)

# 打印指标报告
report = format_metrics_report(metrics)
print(report)

In [ ]:
# Cell 9: 可视化 - 绘制策略回测结果

def plot_strategy_results(df, stock_name, stock_code, backtest_result, buy_hold_result, metrics):
    """
    绘制策略回测结果图表
    """
    # 创建子图：2行1列（主图 + 净值图）
    fig = make_subplots(
        rows=2, cols=1,
        shared_xaxes=True,
        vertical_spacing=0.1,
        row_heights=[0.7, 0.3],
        subplot_titles=(
            f'{stock_name} ({stock_code}) - 均线策略回测',
            '策略净值 vs BUY-HOLD'
        )
    )
    
    # ---- 主图：K线 + 均线 + 交易信号 ----
    
    # K线图
    fig.add_trace(
        go.Candlestick(
            x=df['trade_date'],
            open=df['open'],
            high=df['high'],
            low=df['low'],
            close=df['close'],
            name='股价',
            increasing_line_color='red',   # 中国股市：涨为红
            decreasing_line_color='green',  # 中国股市：跌为绿
        ),
        row=1, col=1
    )
    
    # 短均线
    fig.add_trace(
        go.Scatter(
            x=df['trade_date'],
            y=df['short_ma'],
            mode='lines',
            name=f'短均线({STRATEGY_PARAMS["short_window"]})',
            line=dict(color='orange', width=1.5)
        ),
        row=1, col=1
    )
    
    # 长均线
    fig.add_trace(
        go.Scatter(
            x=df['trade_date'],
            y=df['long_ma'],
            mode='lines',
            name=f'长均线({STRATEGY_PARAMS["long_window"]})',
            line=dict(color='blue', width=1.5)
        ),
        row=1, col=1
    )
    
    # 趋势均线（如果启用）
    if STRATEGY_PARAMS['trend_filter']:
        fig.add_trace(
            go.Scatter(
                x=df['trade_date'],
                y=df['trend_ma'],
                mode='lines',
                name=f'趋势均线({STRATEGY_PARAMS["trend_window"]})',
                line=dict(color='purple', width=1.5, dash='dash')
            ),
            row=1, col=1
        )
    
    # 买入信号
    buy_signals = df[df['signal'] == 1]
    if len(buy_signals) > 0:
        fig.add_trace(
            go.Scatter(
                x=buy_signals['trade_date'],
                y=buy_signals['close'],
                mode='markers',
                name='买入信号',
                marker=dict(
                    symbol='triangle-up',
                    size=12,
                    color='green',
                    line=dict(width=2, color='darkgreen')
                )
            ),
            row=1, col=1
        )
    
    # 卖出信号
    sell_signals = df[df['signal'] == -1]
    if len(sell_signals) > 0:
        fig.add_trace(
            go.Scatter(
                x=sell_signals['trade_date'],
                y=sell_signals['close'],
                mode='markers',
                name='卖出信号',
                marker=dict(
                    symbol='triangle-down',
                    size=12,
                    color='red',
                    line=dict(width=2, color='darkred')
                )
            ),
            row=1, col=1
        )
    
    # ---- 副图：净值曲线对比 ----
    
    fig.add_trace(
        go.Scatter(
            x=df['trade_date'],
            y=backtest_result['nav'],
            mode='lines',
            name='策略净值',
            line=dict(color='blue', width=2)
        ),
        row=2, col=1
    )
    
    fig.add_trace(
        go.Scatter(
            x=df['trade_date'],
            y=buy_hold_result['nav'],
            mode='lines',
            name='BUY-HOLD',
            line=dict(color='gray', width=2, dash='dash')
        ),
        row=2, col=1
    )
    
    # 添加初始资金参考线
    fig.add_hline(
        y=STRATEGY_PARAMS['initial_capital'],
        line_dash='dot',
        line_color='green',
        annotation_text='初始资金',
        row=2, col=1
    )
    
    # ---- 布局设置 ----
    
    fig.update_layout(
        title=dict(
            text=f'{stock_name} ({stock_code}) - 均线策略回测',
            x=0.5,
            font=dict(size=20)
        ),
        xaxis_title='日期',
        yaxis_title='价格（元）',
        yaxis2_title='净值（元）',
        template='plotly_white',
        height=800,
        showlegend=True,
        legend=dict(
            orientation='h',
            yanchor='bottom',
            y=1.02,
            xanchor='right',
            x=1
        ),
        hovermode='x unified'
    )
    
    fig.update_xaxes(rangeslider_visible=False)
    
    return fig

# 绘制图表
print('生成可视化图表...')
fig = plot_strategy_results(df, STOCK_NAME, STOCK_CODE, backtest_result, buy_hold_result, metrics)
fig.show()

# 保存HTML
output_dir = 'outputs/ma_backtest/charts'
os.makedirs(output_dir, exist_ok=True)
output_path = os.path.join(output_dir, f'{STOCK_NAME}_{STOCK_CODE}_backtest.html')
fig.write_html(output_path)
print(f'图表已保存: {output_path}')

In [ ]:
# Cell 10: 批量回测 - 对所有5只股票执行回测

print('=' * 80)
print('批量回测 - 5只股票')
print('=' * 80)

# 股票列表
STOCKS = [
    {'ts_code': '300750.SZ', 'name': '宁德时代', 'industry': '科技/新能源'},
    {'ts_code': '601318.SH', 'name': '中国平安', 'industry': '金融'},
    {'ts_code': '600519.SH', 'name': '贵州茅台', 'industry': '消费'},
    {'ts_code': '601857.SH', 'name': '中国石油', 'industry': '能源'},
    {'ts_code': '002594.SZ', 'name': '比亚迪', 'industry': '制造'}
]

# 存储所有股票的回测结果
all_results = []

# 对每只股票执行回测
for stock in STOCKS:
    print(f'\n处理股票: {stock["name"]} ({stock["ts_code"]})')
    
    try:
        # 加载数据
        df = load_stock_data(
            stock['ts_code'],
            start_date=STRATEGY_PARAMS['start_date'],
            end_date=STRATEGY_PARAMS['end_date']
        )
        
        # 计算指标和信号
        df = generate_signals(df, STRATEGY_PARAMS)
        
        # 执行回测
        backtest_result = run_backtest(df, STRATEGY_PARAMS)
        buy_hold_result = run_buy_hold_backtest(df, STRATEGY_PARAMS)
        
        # 计算指标
        metrics = calculate_metrics(backtest_result, buy_hold_result, df)
        
        # 存储结果
        all_results.append({
            'stock_info': stock,
            'df': df,
            'backtest_result': backtest_result,
            'buy_hold_result': buy_hold_result,
            'metrics': metrics
        })
        
        print(f'  完成: 策略收益率 {metrics["strategy_total_return"]*100:.2f}%')
        
    except Exception as e:
        print(f'  错误: {e}')

print('\n批量回测完成！')

In [ ]:
# Cell 11: 生成汇总对比表

print('=' * 80)
print('汇总对比表')
print('=' * 80)

# 构建汇总数据
summary_data = []
for result in all_results:
    metrics = result['metrics']
    summary_data.append({
        '股票': result['stock_info']['name'],
        '行业': result['stock_info']['industry'],
        '策略收益率(%)': f'{metrics["strategy_total_return"]*100:.2f}',
        'BUY-HOLD收益率(%)': f'{metrics["bh_total_return"]*100:.2f}',
        '超额收益(%)': f'{metrics["excess_return"]*100:.2f}',
        '策略夏普比率': f'{metrics["strategy_sharpe"]:.2f}',
        'BUY-HOLD夏普比率': f'{metrics["bh_sharpe"]:.2f}',
        '策略最大回撤(%)': f'{metrics["strategy_max_drawdown"]*100:.2f}',
        '交易次数': metrics['num_trades']
    })

summary_df = pd.DataFrame(summary_data)
print(summary_df.to_string(index=False))

# 保存汇总表
reports_dir = 'outputs/ma_backtest/reports'
os.makedirs(reports_dir, exist_ok=True)
summary_path = os.path.join(reports_dir, 'summary_report.csv')
summary_df.to_csv(summary_path, index=False, encoding='utf-8-sig')
print(f'\n汇总报告已保存: {summary_path}')

In [ ]:
# Cell 12: 参数优化示例（可选）

print('参数优化示例：测试不同的短均线和长均线组合')
print('=' * 80)

# 定义参数网格
param_grid = [
    {'short_window': 5, 'long_window': 15},
    {'short_window': 5, 'long_window': 20},
    {'short_window': 10, 'long_window': 20},
    {'short_window': 10, 'long_window': 30},
]

# 使用贵州茅台进行参数测试
test_stock = {'ts_code': '600519.SH', 'name': '贵州茅台', 'industry': '消费'}

optimization_results = []

for params in param_grid:
    print(f'\n测试参数: short={params["short_window"]}, long={params["long_window"]}')
    
    # 更新策略参数
    test_params = STRATEGY_PARAMS.copy()
    test_params['short_window'] = params['short_window']
    test_params['long_window'] = params['long_window']
    
    # 加载数据
    df = load_stock_data(
        test_stock['ts_code'],
        start_date=STRATEGY_PARAMS['start_date'],
        end_date=STRATEGY_PARAMS['end_date']
    )
    
    # 计算指标和信号
    df = generate_signals(df, test_params)
    
    # 执行回测
    backtest_result = run_backtest(df, test_params)
    buy_hold_result = run_buy_hold_backtest(df, test_params)
    
    # 计算指标
    metrics = calculate_metrics(backtest_result, buy_hold_result, df)
    
    # 存储结果
    optimization_results.append({
        'short_window': params['short_window'],
        'long_window': params['long_window'],
        'total_return': metrics['strategy_total_return'],
        'sharpe': metrics['strategy_sharpe'],
        'max_drawdown': metrics['strategy_max_drawdown'],
        'num_trades': metrics['num_trades']
    })
    
    print(f'  总收益率: {metrics["strategy_total_return"]*100:.2f}%')
    print(f'  夏普比率: {metrics["strategy_sharpe"]:.2f}')

# 转换为DataFrame并显示
opt_df = pd.DataFrame(optimization_results)
print('\n参数优化结果：')
print(opt_df.to_string(index=False))